### Data Preprocessing and Formatting

In [ ]:
df = pd.read_csv('/content/symptoms.csv')
df.head()

,disease_tag,implicit_symptoms,explicit_symptoms
0,46,"{True: [60], False: []}","{True: array([20]), False: []}"
1,156,"{True: [134], False: []}","{True: array([337]), False: []}"
2,266,"{True: [203], False: []}","{True: array([219]), False: []}"
3,135,"{True: [58, 99, 140], False: []}","{True: array([17, 82, 50]), False: []}"
4,182,"{True: [225], False: []}","{True: array([291, 39]), False: []}"


In [ ]:
pdf = pd.read_csv('/content/disease_symptoms.csv')
pdf.head(30)

,itching,skin_rash,nodal_skin_eruptions,continuous_sneezing,shivering,chills,joint_pain,stomach_pain,acidity,ulcers_on_tongue,...,blackheads,scurring,skin_peeling,silver_like_dusting,small_dents_in_nails,inflammatory_nails,blister,red_sore_around_nose,yellow_crust_ooze,prognosis
0,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal Infection
1,0,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal Infection
2,1,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal Infection
3,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal Infection
4,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal Infection
5,0,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal Infection
6,1,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal Infection
7,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal Infection
8,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal Infection
9,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal Infection


In [ ]:
import pandas as pd

# Load the dataset
file_path = "/content/disease_symptoms.csv"  # Replace with your actual CSV file path
df = pd.read_csv(file_path)

# Separate symptoms and prognosis columns
symptom_columns = df.columns[:-1]  # All columns except the last one (which is 'prognosis')
prognosis_column = df.columns[-1]  # The last column (disease name)

# Dictionary to store the new format
disease_symptoms = {}

# Iterate over each row
for _, row in df.iterrows():
    disease = row[prognosis_column]  # Extract disease name
    symptoms = [symptom for symptom in symptom_columns if row[symptom] == 1]  # Get symptoms with '1'

    # Append symptoms to the disease in dictionary
    if disease in disease_symptoms:
        disease_symptoms[disease].update(symptoms)  # Use a set to avoid duplicates
    else:
        disease_symptoms[disease] = set(symptoms)

# Convert dictionary to a DataFrame
formatted_data = pd.DataFrame({"Disease": disease_symptoms.keys(),
                               "Symptoms": [", ".join(symptoms) for symptoms in disease_symptoms.values()]})

# Save to CSV
output_file = "formatted_disease_symptoms.csv"
formatted_data.to_csv(output_file, index=False)

print(f"Transformed data saved to {output_file}")


Transformed data saved to formatted_disease_symptoms.csv


In [1]:
import pandas as pd
fd = pd.read_csv('formatted_disease_symptoms.csv')
fd.head()

,Disease,Symptoms
0,Fungal Infection,"dischromic _patches, skin_rash, itching, nodal..."
1,Allergy,"watering_from_eyes, shivering, chills, continu..."
2,GERD,"chest_pain, vomiting, ulcers_on_tongue, acidit..."
3,Chronic Cholestasis,"itching, vomiting, loss_of_appetite, yellowing..."
4,Drug Reaction,"skin_rash, itching, burning_micturition, spott..."


### Construction of Knowledge Graph

In [2]:
%pip install llama-index groq chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.7/109.7 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.6/278.6 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 4.4 MB/s eta 0:00:

In [3]:
%pip install llama-index-llms-groq llama-index-embeddings-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 57.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [4]:
import os
from google.colab import userdata
os.environ['GROQ_API_KEY'] = userdata.get('groq_api')
os.environ['LLAMA_CLOUD_API_KEY'] = userdata.get('llama_index_api')

In [16]:
from llama_index.core import SimpleDirectoryReader, KnowledgeGraphIndex,Document,Settings
from llama_index.core.graph_stores import SimpleGraphStore
from llama_index.core import StorageContext
from llama_index.llms.groq import Groq
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings
from IPython.display import Markdown, display
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import KnowledgeGraphIndex, Document, Settings
from llama_index.core.graph_stores import SimpleGraphStore
from llama_index.core import StorageContext
from llama_index.llms.groq import Groq

# Convert CSV to text format
csv_text = fd.to_string(index=False)

# Create a list of Document objects
document = [Document(text=csv_text)]  # Wrap in a list

# Use HuggingFace's embedding model (change model_name if needed)
embedding_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

# Set the embedding model
Settings.embed_model = embedding_model

# Initialize Groq LLM
llm = Groq(model="mixtral-8x7b-32768")  # Replace with the correct Groq model
Settings.llm = llm
Settings.chunk_size = 128

graph_store = SimpleGraphStore()
storage_context = StorageContext.from_defaults(graph_store=graph_store)

# Build the Knowledge Graph Index
index = KnowledgeGraphIndex.from_documents(
    documents=document,  # Ensure it's a list
    max_triplets_per_chunk=2,
    storage_context=storage_context,
)

print("Knowledge Graph Created Successfully with HuggingFace Embedding!")


Knowledge Graph Created Successfully with HuggingFace Embedding!


In [ ]:
index

In [7]:
%pip install llama-index-graph-stores-kuzu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 86.0 MB/s eta 0:00:00


In [8]:
import kuzu

db = kuzu.Database("test1")

In [11]:
from llama_index.graph_stores.kuzu import KuzuGraphStore

graph_store = KuzuGraphStore(db)
storage_context = StorageContext.from_defaults(graph_store=graph_store)

In [18]:
csv_text = fd.to_string(index=False)

# Create a list of Document objects
document = [Document(text=csv_text)]  # Wrap in a list

# Use HuggingFace's embedding model (change model_name if needed)
embedding_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

# Set the embedding model
Settings.embed_model = embedding_model

# Initialize Groq LLM
llm = Groq(model="mixtral-8x7b-32768")  # Replace with the correct Groq model
Settings.llm = llm
Settings.chunk_size = 128


# Build the Knowledge Graph Index
index = KnowledgeGraphIndex.from_documents(
    documents=document,  # Ensure it's a list
    max_triplets_per_chunk=2,
    storage_context=storage_context,
)

print("Knowledge Graph Created Successfully with HuggingFace Embedding!")

Knowledge Graph Created Successfully with HuggingFace Embedding!


In [ ]:
#Loading the Constructed knowledge graph, so that it saves recomputation
temp_index = KnowledgeGraphIndex(nodes=[], storage_context=storage_context)

<ipython-input-44-b81520c9687e>:1: DeprecationWarning: Call to deprecated class KnowledgeGraphIndex. (The KnowledgeGraphIndex class has been deprecated. Please use the new PropertyGraphIndex class instead. If a certain graph store integration is missing in the new class, please open an issue on the GitHub repository or contribute it!) -- Deprecated since version 0.10.53.
  temp_index = KnowledgeGraphIndex(nodes=[], storage_context=storage_context)


### Storing the Knowledge Graph

In [19]:
import networkx as nx

# Assuming `index` is your KnowledgeGraphIndex instance
# Convert the knowledge graph to a NetworkX graph
nx_graph = index.get_networkx_graph()

# Write the NetworkX graph to a GraphML file
nx.write_graphml(nx_graph, "knowledge_graph_symptoms.graphml")

print("Knowledge graph has been successfully exported to 'knowledge_graph.graphml'.")


Knowledge graph has been successfully exported to 'knowledge_graph.graphml'.


In [20]:
import networkx as nx

# Read the GraphML file
graph = nx.read_graphml("knowledge_graph_symptoms.graphml")

# Display the nodes
print("Nodes in the graph:")
for node in graph.nodes(data=True):
    print(node)

# Display the edges
print("\nEdges in the graph:")
for edge in graph.edges(data=True):
    print(edge)


Nodes in the graph:
('Fungal infection', {})
('Dischromic patches', {})
('Skin rash', {})
('Itching', {})
('Nodal skin eruptions', {})
('Drug reaction', {})
('Allergy', {})
('Chronic cholestasis', {})
('Peptic ulcer disease', {})
('Jaundice', {})
('Chickenpox', {})
('Liver disease', {})
('Watering from eyes', {})
('Shivering', {})
('Chills', {})
('Continuous sneezing', {})
('Watering\\_from\\_eyes', {})
('Continuous\\_sneezing', {})
('Gerd', {})
('Malaria', {})
('Dengue', {})
('Typhoid', {})
('Tuberculosis', {})
('Common cold', {})
('Pneumonia', {})
('Heart attack', {})
('Chest pain', {})
('Vomiting', {})
('Ulcers on tongue', {})
('Acidity', {})
('Stomach pain', {})
('Cough', {})
('Chest\\_pain', {})
('Ulcers\\_on\\_tongue', {})
('Stomach\\_pain', {})
('Gastroenteritis', {})
('Bronchial asthma', {})
('Paralysis', {})
('Paralysis (brain hemorrhage', {})
('Brain hemorrhage', {})
('Alcoholic hepatitis', {})
('Hypothyroidism', {})
('Hypoglycemia', {})
('Vertigo', {})
('Migraine', {})
('Cer

In [21]:
# Assuming `index` is your KnowledgeGraphIndex instance
temp_graph_store = index.graph_store


In [ ]:
## create graph
from pyvis.network import Network

g = index.get_networkx_graph()
net = Network(notebook=True, cdn_resources="in_line", directed=True)
net.from_nx(g)
net.show("kuzugraph_draw.html")

kuzugraph_draw.html


### Accessing the Attributes of the Constructed Knowledge Graph

In [ ]:
from llama_index.graph_stores.kuzu import KuzuGraphStore
import networkx as nx

# Assuming `index` is your KnowledgeGraphIndex instance
# Convert the knowledge graph to a NetworkX graph
nx_graph = index.get_networkx_graph()

# Iterate over nodes
print("Nodes in the Knowledge Graph:")
for node in nx_graph.nodes(data=True):
    print(node)

# Iterate over edges
print("\nEdges in the Knowledge Graph:")
for edge in nx_graph.edges(data=True):
    print(edge)


Nodes in the Knowledge Graph:
('Fungal infection', {})
('Dischromic patches', {})
('Skin rash', {})
('Itching', {})
('Nodal skin eruptions', {})
('Drug reaction', {})
('Allergy', {})
('Chronic cholestasis', {})
('Peptic ulcer disease', {})
('Jaundice', {})
('Malaria', {})
('Chickenpox', {})
('Liver disease', {})
('Watering from eyes', {})
('Shivering', {})
('Chills', {})
('Continuous sneezing', {})
('Watering\\_from\\_eyes', {})
('Continuous\\_sneezing', {})
('Gerd', {})
('Dengue', {})
('Typhoid', {})
('Tuberculosis', {})
('Common cold', {})
('Pneumonia', {})
('Chest pain', {})
('Vomiting', {})
('Ulcers on tongue', {})
('Acidity', {})
('Stomach pain', {})
('Cough', {})
('Chest\\_pain', {})
('Ulcers\\_on\\_tongue', {})
('Stomach\\_pain', {})
('Gastroenteritis', {})
('Bronchial asthma', {})
('Paralysis', {})
('Paralysis (brain hemorrhage', {})
('Brain hemorrhage', {})
('Alcoholic hepatitis', {})
('Heart attack', {})
('Hypothyroidism', {})
('Hypoglycemia', {})
('Vertigo', {})
('Migraine',

### Testing Out Tavily Search

In [22]:
%pip install llama-index chromadb phidata tavily

ERROR: Could not find a version that satisfies the requirement tavily (from versions: none)
ERROR: No matching distribution found for tavily


In [23]:
%pip install -qU "langchain-community>=0.2.11" tavily-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 56.5 MB/s eta 0:00:00


In [24]:
os.environ['TAVILY_API_KEY'] = userdata.get('tavily_api')

In [25]:
from tavily import TavilyClient

# Step 1. Instantiating your TavilyClient
tavily_client = TavilyClient()

# Step 2. Executing a simple search query
response = tavily_client.search("What is an Agentic RAG?")

# Step 3. That's it! You've done a Tavily Search!
print(response['results'][2]['content'])

What Is Agentic RAG? Agentic RAG (Agent-based RAG implementation) revolutionizes question answering through an innovative agent-based framework. Unlike traditional approaches that solely rely on large language models (LLMs), agentic RAG employs intelligent agents to adeptly tackle complex questions.


### Creating the Agent

In [26]:
%pip install -U phidata

  Using cached phidata-2.7.10-py3-none-any.whl.metadata (38 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 716.9/716.9 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.0/236.0 kB 18.8 MB/s eta 0:00:00


In [27]:
os.environ['GEMINI_API_KEY'] = userdata.get('gemini_api')

In [31]:
os.environ['GOOGLE_API_KEY'] = userdata.get('google_api')

In [29]:
from phi.agent import Agent
from phi.model.google import Gemini
from phi.tools.tavily import TavilyTools
import networkx as nx

# Assuming `index` is your KnowledgeGraphIndex instance
# Convert the knowledge graph to a NetworkX graph
nx_graph = index.get_networkx_graph()

# Initialize a list to store the string representation
graph_str_list = []

# Add nodes to the string representation
graph_str_list.append("Nodes:\n")
for node in nx_graph.nodes(data=True):
    graph_str_list.append(f"{node}\n")

# Add edges to the string representation
graph_str_list.append("\nEdges:\n")
for edge in nx_graph.edges(data=True):
    graph_str_list.append(f"{edge}\n")

# Join the list into a single string
knowledge_graph_text = "".join(graph_str_list)

# Define the AI Agent with system-level knowledge
agent = Agent(
    model=Gemini(id="gemini-1.5-flash"),  # Using Gemini 1.5-flash as the LLM
    tools=[TavilyTools()],  # Web search for medical recommendations
    description="An AI-powered symptom analyzer that provides differential diagnoses and recommended medical actions.",
    instructions=[
        "You are an AI medical assistant that provides users with a preliminary differential diagnosis and recommended medical actions.",
        "You have access to a medical Knowledge Graph that maps symptoms to possible conditions.",
        "You also have access to the Tavily search tool to retrieve real-time medical advice and recommendations.",
        "When given symptoms, follow these steps:",
        "1. Identify possible medical conditions based on the Knowledge Graph.",
        "2. Use Tavily to retrieve recommended actions (e.g., tests, treatments, or doctor consultations).",
        "3. Generate a concise response combining both aspects, ensuring clarity in about 2-3 sentences.",
        "Here is the structured medical Knowledge Graph data you should use:\n\n"
        f"{knowledge_graph_text}\n\n"
        "Use this knowledge before making any diagnoses."
    ],
    markdown=True,
    show_tool_calls=True,
)

print("AI Agent initialized with the knowledge graph.")


AI Agent initialized with the knowledge graph.


In [32]:
def analyze_symptoms(user_input):
    # Generate response using both Knowledge Graph (from system prompt) and Tavily search
    final_response = agent.print_response(user_input)
    return final_response

# Example usage
symptom_input = "I have been experiencing chest pain and shortness of breath."
response = analyze_symptoms(symptom_input)
print(response)

┏━ Message ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                  ┃
┃ I have been experiencing chest pain and shortness of breath.                                     ┃
┃                                                                                                  ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Response (3.6s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                  ┃
┃ Based on your symptoms of chest pain and shortness of breath, possible conditions include a      ┃
┃ heart attack, pneumonia, or a severe case of GERD.  I recommend seeking immediate medical        ┃
┃ attention to get a proper diagnosis and treatment plan.  To get more specific advice, I s

In [33]:
symptom_input = "I have been feeling fatigued, experiencing frequent headaches, and occasional dizziness."
response = analyze_symptoms(symptom_input)
print(response)

┏━ Message ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                  ┃
┃ I have been feeling fatigued, experiencing frequent headaches, and occasional dizziness.         ┃
┃                                                                                                  ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Response (3.9s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                  ┃
┃ Based on your symptoms of fatigue, headaches, and dizziness, several conditions are possible,    ┃
┃ including migraine, hypoglycemia, or even cervical spondylosis.  To determine the exact cause, I ┃
┃ recommend consulting a doctor for a proper diagnosis and to discuss appropriate treatment